# Customer Support Dataset Notebook

This notebook performs data cleaning, preprocessing, and dataset splitting for training a customer support intent classification model.

**Dataset:** Bitext Sample Customer Support Training Dataset (27K responses, v11)  
**Goal:** Produce a clean, deduplicated, and properly split dataset ready for intent classification model training.

---

### Pipeline Overview
| Step | Description |
|------|-------------|
| 1 | Load the raw CSV and select relevant columns |
| 2 | Drop rows with missing values |
| 3 | Normalize and clean text |
| 4 | Remove duplicate entries |
| 5 | Filter by token length |
| 6 | Inspect intent class distribution |
| 7 | Remove intents with insufficient samples |
| 8 | Final duplicate safety check |
| 9 | Save cleaned dataset |
| 10–11 | Stratified train / validation / test split and save |
| 12 | Final sanity check |

In [1]:
import os
import pandas as pd
import re
from sklearn.model_selection import train_test_split

**Libraries used:**
- `os` — file path checks and directory creation
- `pandas` — data loading, manipulation, and CSV I/O
- `re` — regular expressions for text normalization
- `sklearn.model_selection.train_test_split` — stratified dataset splitting

## 1. Load Dataset

Reads the raw CSV file from the `dataset/` directory. Only the `instruction` (renamed to `text`) and `intent` columns are retained — all other columns are discarded to keep the DataFrame lean.

In [2]:
file_path = "dataset/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Dataset not found at {file_path}")

df = pd.read_csv(file_path)

# Keep only needed columns
df = df[["instruction", "intent"]].copy()
df.rename(columns={"instruction": "text", "intent": "intent"}, inplace=True)

print("Original Dataset Shape:", df.shape)

Original Dataset Shape: (26872, 2)


## 2. Remove Missing Values

Drops any row where either `text` or `intent` is null. Null entries cannot be used for training and would cause errors in downstream processing.

In [3]:
df.dropna(inplace=True)
print("After removing NULLs:", df.shape)

After removing NULLs: (26872, 2)


## 3. Text Cleaning

Applies a normalization function to each `text` entry:
- Converts to **lowercase** and strips leading/trailing whitespace
- Collapses multiple whitespace characters into a single space
- Removes all characters **except** alphanumerics, spaces, `?`, and `!` — punctuation useful for intent detection is preserved

Intent labels are also lowercased and stripped to ensure consistent formatting.

In [4]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)  # remove extra spaces
    # keep useful punctuation for intent detection
    text = re.sub(r'[^a-z0-9\s\?\!]', '', text)
    return text

df["text"] = df["text"].apply(clean_text)
df["intent"] = df["intent"].astype(str).str.lower().str.strip()

## 4. Remove Duplicates

Removes rows where both `text` and `intent` are identical. Duplicates inflate class counts and can cause data leakage between train and test splits if the same sample appears in both.

In [5]:
before = len(df)
df.drop_duplicates(subset=["text", "intent"], inplace=True)
after = len(df)

print(f"Removed {before - after} duplicate rows")
print("Dataset Shape:", df.shape)

Removed 2687 duplicate rows
Dataset Shape: (24185, 2)


## 5. Text Length Filtering

Computes word count per entry and applies two filters:
- **Minimum length > 0** — removes entries that became empty after cleaning
- **Maximum length ≤ 128 tokens** — removes unusually long entries that fall outside the expected range for customer support queries

The temporary `length` column is dropped after filtering so it does not appear in any output files.

In [6]:
df["length"] = df["text"].apply(lambda x: len(x.split()))

print("\nText Length Stats:")
print(df["length"].describe())

df = df[df["length"] > 0]     # remove empty text
df = df[df["length"] <= 128]  # limit max length

# Drop the helper column so it doesn't appear in outputs
df.drop(columns=["length"], inplace=True)

print("After filtering:", df.shape)


Text Length Stats:
count    24185.000000
mean         8.855324
std          2.563216
min          1.000000
25%          7.000000
50%          9.000000
75%         11.000000
max         16.000000
Name: length, dtype: float64
After filtering: (24185, 2)


## 6. Intent Distribution

Prints the count of samples per intent class. This helps identify class imbalance — classes with very few samples may need to be removed before stratified splitting can be applied.

In [7]:
print("\nIntent Distribution:")
print(df["intent"].value_counts())


Intent Distribution:
intent
complaint                   996
place_order                 992
review                      992
check_payment_methods       991
payment_issue               991
delivery_period             990
newsletter_subscription     990
set_up_shipping_address     989
recover_password            987
contact_customer_service    983
registration_problems       979
contact_human_agent         977
check_refund_policy         974
change_shipping_address     960
check_cancellation_fee      939
check_invoice               913
get_refund                  907
delete_account              897
create_account              874
get_invoice                 874
change_order                854
switch_account              822
track_order                 779
edit_account                765
track_refund                687
delivery_options            655
cancel_order                428
Name: count, dtype: int64


## 7. Remove Rare Intents

Filters out any intent class with only **1 sample**. `train_test_split` with `stratify` requires at least 2 samples per class — one for training and one for the held-out set. Keeping singletons would cause a `ValueError` at split time.

In [8]:
intent_counts = df["intent"].value_counts()
valid_intents = intent_counts[intent_counts > 1].index
df = df[df["intent"].isin(valid_intents)]

print("After removing rare intents:", df.shape)

After removing rare intents: (24185, 2)


## 8. Final Duplicate Check

A safety pass to confirm no duplicates remain after all prior transformations. Near-identical entries that differ only in whitespace or punctuation may have converged to the same string after cleaning, which this step catches.

In [9]:
duplicates = df[df.duplicated(subset=["text", "intent"], keep=False)]
print("Remaining duplicates:", len(duplicates))

Remaining duplicates: 0


## 9. Save Cleaned Dataset

Saves the fully cleaned and filtered DataFrame to `output/cleaned_dataset.csv`. This serves as the master cleaned file before any splitting — useful for inspection or rerunning splits with different ratios.

In [10]:
os.makedirs("output", exist_ok=True)
df.to_csv("output/cleaned_dataset.csv", index=False)
print("Cleaned dataset saved to output/cleaned_dataset.csv")

Cleaned dataset saved to output/cleaned_dataset.csv


## 10. Train / Validation / Test Split

Splits the cleaned dataset into three stratified subsets:
- **Train (80%)** — used to fit the model
- **Validation (10%)** — used for hyperparameter tuning and early stopping
- **Test (10%)** — held out entirely for final evaluation

`stratify=intent` ensures each split reflects the original class distribution. `random_state=42` guarantees reproducibility.

In [11]:
train, temp = train_test_split(
    df,
    test_size=0.2,
    stratify=df["intent"],
    random_state=42
)

val, test = train_test_split(
    temp,
    test_size=0.5,
    stratify=temp["intent"],
    random_state=42
)

print("\nDataset Split Sizes:")
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))


Dataset Split Sizes:
Train: 19348
Validation: 2418
Test: 2419


## 11. Save Splits

Writes each split to a separate CSV file under `output/`. These files are the direct inputs for model training and evaluation pipelines.

In [12]:
train.to_csv("output/train.csv", index=False)
val.to_csv("output/validation.csv", index=False)
test.to_csv("output/test.csv", index=False)
print("Splits saved to output/")

Splits saved to output/


## 12. Final Check

Prints a sample of the cleaned data and confirms there are no remaining null values — a final sanity check before the dataset is handed off to the model training pipeline.

In [13]:
print("\nSample Data:")
print(df.head())

print("\nMissing Values Check:")
print(df.isnull().sum())


Sample Data:
                                                text        intent
0       question about cancelling order order number  cancel_order
1  i have a question about cancelling oorder orde...  cancel_order
2        i need help cancelling puchase order number  cancel_order
3             i need to cancel purchase order number  cancel_order
4  i cannot afford this order cancel purchase ord...  cancel_order

Missing Values Check:
text      0
intent    0
dtype: int64
